In [1]:
# Import necessary modules
import sys
from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use('Agg')

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from database import db
from dashboard import dashboard
from student import student_manager
from attendance import attendance_manager

2026-07-03 11:29:45,220 - database - INFO - Successfully connected to MongoDB database: attendance_system
2026-07-03 11:29:55,475 - keras_facenet.embedding_model - INFO - Loading weights.
2026-07-03 11:29:55,478 - keras_facenet.utils - INFO - Looking for C:\Users\sanja/.keras-facenet\20180402-114759\20180402-114759-weights.h5
2026-07-03 11:30:01,252 - tensorflow - WARNING - From C:\Users\sanja\OneDrive\Desktop\attendance_system\venv\Lib\site-packages\keras\src\backend\tensorflow\core.py:233: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

2026-07-03 11:30:02,361 - embedding_generator - INFO - FaceNet model loaded successfully
2026-07-03 11:30:02,970 - recognition - WARNING - MediaPipe initialization failed: module 'mediapipe.tasks.python.vision.face_detector' has no attribute 'RunningMode'
2026-07-03 11:30:02,973 - recognition - INFO - Falling back to OpenCV Haar Cascade...
2026-07-03 11:30:03,027 - recognition - INFO - Using OpenCV Haar Cascade fro

In [2]:
# Update dashboard statistics
stats = dashboard.update_stats()

print("📊 ATTENDANCE SYSTEM DASHBOARD")
print("=" * 50)
print(f"Last Updated: {stats.get('timestamp', 'N/A')}")
print()

# Students
print("👥 STUDENTS")
print("-" * 30)
for key, value in stats.get('students', {}).items():
    print(f"  {key.replace('_', ' ').title()}: {value}")
print()

# Attendance
print("📈 TODAY'S ATTENDANCE")
print("-" * 30)
for key, value in stats.get('attendance', {}).items():
    if key == "percentage":
        print(f"  {key.title()}: {value:.1f}%")
    else:
        print(f"  {key.title()}: {value}")
print()

# Database
print("💾 DATABASE")
print("-" * 30)
for key, value in stats.get('database', {}).items():
    print(f"  {key.title()}: {value}")
print()

# Embeddings
print("🧠 EMBEDDINGS")
print("-" * 30)
for key, value in stats.get('embeddings', {}).items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

📊 ATTENDANCE SYSTEM DASHBOARD
Last Updated: 2026-07-03 11:30:03

👥 STUDENTS
------------------------------
  Total: 5
  Face Captured: 1
  Embeddings Generated: 1
  Pending Capture: 4
  Pending Embeddings: 0

📈 TODAY'S ATTENDANCE
------------------------------
  Total: 10
  Present: 2
  Absent: 8
  Percentage: 20.0%

💾 DATABASE
------------------------------
  Students: 5
  Embeddings: 1
  Attendance: 10
  Logs: 5

🧠 EMBEDDINGS
------------------------------
  Total Students: 5
  Face Captured: 1
  Embeddings Generated: 1
  Pending: 0


In [3]:
# Generate attendance chart
chart_data = dashboard.get_attendance_chart_data(days=7)

if chart_data and chart_data.get("dates"):

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Bar chart
    x = range(len(chart_data["dates"]))
    width = 0.35

    ax1.bar(
        [i - width / 2 for i in x],
        chart_data["present"],
        width,
        label="Present",
        color="green",
        alpha=0.8,
    )

    ax1.bar(
        [i + width / 2 for i in x],
        chart_data["absent"],
        width,
        label="Absent",
        color="red",
        alpha=0.8,
    )

    ax1.set_xlabel("Date")
    ax1.set_ylabel("Students")
    ax1.set_title("Daily Attendance (Last 7 Days)")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(chart_data["dates"], rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Line Chart
    ax2.plot(
        chart_data["dates"],
        chart_data["percentages"],
        marker="o",
        linewidth=2,
        markersize=8,
        color="blue",
    )

    ax2.set_xlabel("Date")
    ax2.set_ylabel("Attendance (%)")
    ax2.set_title("Attendance Percentage Trend")
    ax2.tick_params(axis="x", rotation=45)
    ax2.grid(True, alpha=0.3)

    ax2.axhline(
        y=75,
        color="red",
        linestyle="--",
        alpha=0.5,
        label="Target (75%)",
    )

    ax2.legend()

    plt.tight_layout()
    plt.show()

else:
    print("No attendance data available.")

2026-07-03 11:30:03,138 - dashboard - ERROR - Error getting chart data: name 'timedelta' is not defined


No attendance data available.


In [4]:
from datetime import timedelta

start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")

print(f"Attendance Statistics ({start_date} → {end_date})")
print("=" * 50)

stats_30d = attendance_manager.get_attendance_statistics(
    start_date,
    end_date,
)

if stats_30d:

    print(f"Total Sessions : {stats_30d.get('total_sessions',0)}")
    print(f"Total Records  : {stats_30d.get('total_records',0)}")
    print(f"Present        : {stats_30d.get('overall_present',0)}")
    print(f"Absent         : {stats_30d.get('overall_absent',0)}")
    print(
        f"Attendance %   : {stats_30d.get('overall_attendance_percentage',0):.1f}%"
    )

    students = stats_30d.get("student_stats", [])

    if students:

        print("\nTop 5 Students")

        top = sorted(
            students,
            key=lambda x: x.get("attendance_percentage", 0),
            reverse=True,
        )[:5]

        for s in top:

            print(
                f"{s['name']} ({s['roll_number']}) "
                f"{s['attendance_percentage']:.1f}% "
                f"({s['present']}/{s['total_sessions']})"
            )

else:
    print("No statistics available.")

Attendance Statistics (2026-06-03 → 2026-07-03)
Total Sessions : 2
Total Records  : 10
Present        : 2
Absent         : 8
Attendance %   : 20.0%

Top 5 Students
Rahul Kumar (22001) 100.0% (2/2)
Priya Sharma (22002) 0.0% (0/2)
Sai Reddy (22003) 0.0% (0/2)
Krishna Kumar (22004) 0.0% (0/2)
Ravi Singh (22005) 0.0% (0/2)


In [5]:
today = datetime.now().strftime("%Y-%m-%d")

today_attendance = attendance_manager.get_attendance_summary(date=today)

print(f"Today's Attendance ({today})")
print("=" * 50)

if today_attendance and today_attendance.get("records"):

    records = today_attendance["records"]

    present = [r for r in records if r["status"] == "present"]
    absent = [r for r in records if r["status"] == "absent"]

    print(f"Present : {len(present)}")
    print(f"Absent  : {len(absent)}")

    if present:

        print("\nPresent Students")

        for student in present[:10]:

            print(
                f"{student['name']} "
                f"({student['roll_number']}) "
                f"{student.get('time_in','N/A')}"
            )

    if absent:

        print("\nAbsent Students")

        for student in absent[:10]:

            print(
                f"{student['name']} "
                f"({student['roll_number']})"
            )

else:

    print("No attendance found for today.")

Today's Attendance (2026-07-03)
Present : 2
Absent  : 8

Present Students
Rahul Kumar (22001) 11:23:13
Rahul Kumar (22001) 11:27:33

Absent Students
Priya Sharma (22002)
Sai Reddy (22003)
Krishna Kumar (22004)
Ravi Singh (22005)
Priya Sharma (22002)
Sai Reddy (22003)
Krishna Kumar (22004)
Ravi Singh (22005)


In [6]:
import pandas as pd

data = []

students = student_manager.get_all_students()

for student in students:

    roll = student["roll_number"]

    records = attendance_manager.get_attendance_by_student(roll)

    if records:

        total = len(records)
        present = sum(
            1 for r in records
            if r["status"] == "present"
        )

        percentage = (present / total) * 100

    else:

        total = 0
        present = 0
        percentage = 0

    data.append({
        "Roll Number": roll,
        "Name": student["name"],
        "Department": student["department"],
        "Section": student["section"],
        "Total Sessions": total,
        "Present": present,
        "Attendance Percentage": round(percentage, 2)
    })

if data:

    df = pd.DataFrame(data)

    df = df.sort_values(
        "Attendance Percentage",
        ascending=False
    )

    filename = (
        f"dashboard_export_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )

    filepath = Path("../Attendance") / filename

    df.to_csv(filepath, index=False)

    print(f"✓ Exported to {filepath}")

    print("\nPreview")
    print(df.head(10))

else:

    print("No data available.")

✓ Exported to ..\Attendance\dashboard_export_20260703_113003.csv

Preview
  Roll Number           Name        Department Section  Total Sessions  \
0       22001    Rahul Kumar  Computer Science       A               2   
1       22002   Priya Sharma  Computer Science       A               2   
2       22003      Sai Reddy  Computer Science       A               2   
3       22004  Krishna Kumar  Computer Science       A               2   
4       22005     Ravi Singh  Computer Science       A               2   

   Present  Attendance Percentage  
0        2                  100.0  
1        0                    0.0  
2        0                    0.0  
3        0                    0.0  
4        0                    0.0  


In [7]:
choice = input(
    "\nRun Live Dashboard with Camera Feed? (y/n): "
).strip().lower()

if choice == "y":

    print("\nStarting Dashboard...")
    print("Press 'q' to exit.")

    dashboard.run_dashboard()


Run Live Dashboard with Camera Feed? (y/n):  y



Starting Dashboard...
Press 'q' to exit.


2026-07-03 11:31:32,019 - dashboard - INFO - Starting dashboard. Press 'q' to quit.


In [8]:
db.close()

print("\n✓ Database connection closed")

2026-07-03 11:33:27,217 - database - INFO - MongoDB connection closed



✓ Database connection closed
